# 📧 Option 3 — AI Email Generator
### AI Engineer Ready — Cohort 1 | Assignment 1

---

## 🗺️ Project Overview

In this assignment, you will build an **AI-powered email generator** that:
- Takes a short **context** (e.g. "Schedule a meeting about Project X")
- Takes a **tone** (formal / informal / neutral)
- Uses an **LLM (Large Language Model)** via LangChain's `PromptTemplate` to generate a full, well-written email

By the end of this notebook, you will have a working function `generate_email(context, tone)` that produces real emails using AI — no more placeholder text!

---

## 📁 Files in This Project

| File | Purpose |
|---|---|
| `AI_Email_Generator.ipynb` | **This notebook** — your main workspace |
| `main.py` | Command-line version of the same logic |
| `requirements.txt` | Python packages needed (`openai`, `langchain`) |
| `README.md` | Short project description |

---

## ✅ What You'll Complete (Step by Step)

| Step | Task | Status |
|---|---|---|
| 1 | Install dependencies | ⬜ TODO |
| 2 | Set up your OpenAI API key | ⬜ TODO |
| 3 | Understand `PromptTemplate` | ⬜ TODO |
| 4 | Build the `generate_email()` function | ⬜ TODO |
| 5 | Test with multiple tones | ⬜ TODO |
| 6 | Update `main.py` with your implementation | ⬜ TODO |
| 7 | Bonus: Multi-language support | ⬜ BONUS |

---

> 💡 **Tip:** Run each cell top to bottom. Read every markdown cell — they explain *why* you're doing each step, not just *what* to do.

---
## Step 1 — 📦 Install Dependencies

### 📖 README

Before writing any code, you need to install the required Python libraries listed in `requirements.txt`:

- **`openai`** — The official OpenAI Python SDK. LangChain uses this under the hood to talk to GPT models.
- **`langchain`** — A framework that makes it easy to chain together LLM calls, prompts, and tools.
- **`langchain-openai`** — The LangChain integration specifically for OpenAI models.

### 🎯 What to do
Run the cell below. You should see output like `Successfully installed openai-x.x.x langchain-x.x.x`.

### 📊 How you'll be evaluated
- The cell must run without errors.
- All three packages must be installed.

In [1]:
# TODO: Run this cell to install all required packages
# You only need to run this ONCE. After that, you can comment it out.

%pip install openai langchain langchain-openai --quiet

print("✅ All packages installed successfully!")

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed successfully!



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Step 2 — 🔑 Set Up Your OpenAI API Key

### 📖 README

To use GPT models, you need an **API key** from OpenAI. Think of it like a password that lets your code talk to OpenAI's servers.

**How to get an API key:**
1. Go to [https://platform.openai.com/api-keys](https://platform.openai.com/api-keys)
2. Sign in (or create a free account)
3. Click **"Create new secret key"**
4. Copy the key — it looks like `sk-...`

**Security rule ⚠️:** Never hardcode your API key in code you'll share or push to GitHub. We use `getpass` here so it's only stored in memory during this session.

### 🎯 What to do
Run the cell below. A password prompt will appear. Paste your OpenAI API key there.

### 📊 How you'll be evaluated
- The environment variable `OPENAI_API_KEY` must be set.
- The key must be valid (later steps will fail with an `AuthenticationError` if it's wrong).
- Your submitted notebook must **NOT** contain your actual API key as plain text.

In [ ]:
import os
import getpass

# TODO: Run this cell and paste your OpenAI API key when prompted
# The key will NOT be stored in the notebook — it only lives in memory

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter your OpenAI API Key: ")

# Verify the key is set (prints masked version)
key = os.environ.get("OPENAI_API_KEY", "")
if key.startswith("sk-"):
    print(f"✅ API key loaded: sk-...{key[-4:]}")
else:
    print("❌ API key not found or looks wrong. Check that it starts with 'sk-'")

---
## Step 3 — 🧩 Understand PromptTemplate

### 📖 README

A **PromptTemplate** is a reusable prompt with **placeholders** (variables) that get filled in at runtime — like a Mad Libs for AI instructions.

**Example:**
```
Template:  "Write a {tone} email about {context}."
Variables: tone = "formal", context = "project deadline"
Result:    "Write a formal email about project deadline."
```

Why use `PromptTemplate` instead of just an f-string?
- ✅ Reusable and clean
- ✅ Easy to test with different inputs
- ✅ Works natively with LangChain chains
- ✅ Handles special characters safely

### 🎯 What to do
Read and run the example cell below. Try changing the values to see how the template fills in.

### 📊 How you'll be evaluated
- You understand the concept well enough to write your own template in Step 4.
- Your template in Step 4 must use `PromptTemplate` (not just an f-string).

In [ ]:
# This is a DEMO cell — read it carefully before moving to Step 4
from langchain_core.prompts import PromptTemplate


# --- EXAMPLE: How PromptTemplate works ---

example_template = PromptTemplate(
    input_variables=["tone", "context"],
    template="Write a {tone} email about: {context}"
)

# Fill in the variables
filled_prompt = example_template.format(tone="formal", context="scheduling a meeting")
print("📝 Filled prompt:")
print(filled_prompt)
print()

# Try it with a different tone
filled_prompt_2 = example_template.format(tone="informal", context="birthday party invitation")
print("📝 Another filled prompt:")
print(filled_prompt_2)

# TODO (optional): Try changing the context and tone values above and re-run to see the output change

---
## Step 4 — 🛠️ Build the `generate_email()` Function  ⭐ MAIN TASK

### 📖 README

This is the **core task** of the assignment. You will implement the `generate_email(context, tone)` function that:

1. **Creates a `PromptTemplate`** — Write a detailed prompt that instructs the LLM to write a complete email. Your prompt should tell the model:
   - What tone to use (`formal`, `informal`, or `neutral`)
   - What the email is about (the `context`)
   - To include: a subject line, greeting, body, and sign-off

2. **Initialises the LLM** — Use `ChatOpenAI` from `langchain_openai`. Use model `"gpt-3.5-turbo"` (cheaper) or `"gpt-4o-mini"`.

3. **Creates a chain** — Connect the prompt template to the LLM using `|` (the LangChain pipe operator).

4. **Invokes the chain** — Call `.invoke({"context": context, "tone": tone})` and return the result.

### 📊 How you'll be evaluated
| Criteria | Marks |
|---|---|
| Uses `PromptTemplate` with correct `input_variables` | 20% |
| Prompt is detailed and instructs for subject + body + sign-off | 20% |
| `ChatOpenAI` is correctly initialised | 20% |
| Chain is created using `prompt \| llm` pattern | 20% |
| Function returns a complete, readable email string | 20% |

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# ============================================================
# TODO: Implement the generate_email function below
# Read the README in the cell above CAREFULLY before starting
# ============================================================

def generate_email(context: str, tone: str = "neutral") -> str:
    """
    Generates a complete email using an LLM.

    Args:
        context (str): What the email should be about.
                       Example: "Schedule a meeting about Q3 results"
        tone (str):    The writing tone. One of: 'formal', 'informal', 'neutral'

    Returns:
        str: A complete email with subject line, greeting, body, and sign-off.
    """

    # TODO Step 4a: Define your PromptTemplate
    # Hint: Your template should clearly instruct the model to:
    #   - Write an email in the given {tone}
    #   - Cover the given {context}
    #   - Include: Subject line, Greeting, Email body (2-3 paragraphs), Sign-off
    prompt = PromptTemplate(
        input_variables=["context", "tone"],
        template="""TODO: Write your prompt here.
        
Use the variables {context} and {tone} in your template."""
    )

    # TODO Step 4b: Initialise the ChatOpenAI LLM
    llm = None  # TODO: Replace None with ChatOpenAI(...)

    # TODO Step 4c: Create the chain by connecting prompt and llm
    chain = None  # TODO: Replace None with the chain

    # TODO Step 4d: Invoke the chain with context and tone, return the result
    
    return "TODO: Replace this string with your chain invocation result"


print("✅ Function defined! Move to Step 5 to test it.")

---
## Step 5 — 🧪 Test With Multiple Tones

### 📖 README

Testing is a critical part of software development. You should verify that:
- Your function runs without errors
- The output actually looks like a proper email (subject, greeting, body, sign-off)
- The **tone changes** the writing style:
  - `formal` → professional language, "Dear Sir/Madam", structured paragraphs
  - `informal` → casual language, "Hey!", conversational style
  - `neutral` → balanced, neither too stiff nor too casual

### 🎯 What to do
Run each sub-cell below. Read the output and check if the tone actually matches what was requested.

### 📊 How you'll be evaluated
| Criteria | Check |
|---|---|
| All 3 test cells run without errors | Required |
| Each email has a Subject, Greeting, Body, Sign-off | Required |
| Formal email sounds noticeably different from informal | Required |
| Emails are relevant to the provided context | Required |

In [ ]:
# TEST 1: Formal tone
# TODO: Run this cell after completing Step 4

print("=" * 60)
print("TEST 1: FORMAL TONE")
print("=" * 60)

email_formal = generate_email(
    context="Schedule a meeting to discuss Q3 financial results with the board",
    tone="formal"
)

print(email_formal)

In [ ]:
# TEST 2: Informal tone
# TODO: Run this cell after completing Step 4

print("=" * 60)
print("TEST 2: INFORMAL TONE")
print("=" * 60)

email_informal = generate_email(
    context="Remind a friend that we're going hiking this weekend",
    tone="informal"
)

print(email_informal)

In [ ]:
# TEST 3: Neutral tone — USE YOUR OWN CONTEXT!
# TODO: Change the context to something meaningful to you

print("=" * 60)
print("TEST 3: NEUTRAL TONE (Your own context)")
print("=" * 60)

# TODO: Change the context below to something of your choice
my_context = "TODO: Write your own context here — be creative!"

email_neutral = generate_email(
    context=my_context,
    tone="neutral"
)

print(email_neutral)

---
## Step 6 — 📋 Update main.py

### 📖 README

Your `main.py` file currently has a **placeholder** `generate_email()` function that doesn't use any AI — it just returns a dummy string. Now that you have a working implementation in this notebook, you need to copy your logic into `main.py` so the command-line version also works.

**Current placeholder in main.py:**
```python
def generate_email(context: str, tone: str = "neutral") -> str:
    # TODO: Replace with PromptTemplate + LLM call
    return f"[{tone.title()} email]\nSubject: {context}\n\nDear Recipient,\n\n(Replace this with generated email body.)"
```

**What you need to do:**
1. Open `main.py` in your code editor
2. Add the necessary imports at the top (`PromptTemplate`, `ChatOpenAI`, `os`)
3. Replace the placeholder `generate_email()` function body with your working implementation from Step 4

---
## 🏁 Submission Checklist

Before submitting, go through this checklist:

- [ ] **Step 1** — Dependencies installed, cell runs without errors
- [ ] **Step 2** — API key loaded (notebook does NOT contain the raw key)
- [ ] **Step 3** — PromptTemplate demo cell was run and understood
- [ ] **Step 4** — `generate_email()` implemented with `PromptTemplate` + `ChatOpenAI` + chain
- [ ] **Step 5** — All 3 test cells run and produce real emails (formal / informal / neutral)
- [ ] **Step 6** — `main.py` updated and tested from terminal

---

## 📤 What to Submit

1. This notebook (`AI_Email_Generator.ipynb`) with all cells **run** (output visible)
2. Your updated `main.py`
3. A screenshot or copy-paste of the terminal output when running `main.py`

---

## 💬 Need Help?

- Re-read the README cells — they contain hints and skeleton code
- Check the [LangChain PromptTemplate docs](https://python.langchain.com/docs/concepts/prompt_templates/)
- Check the [LangChain ChatOpenAI docs](https://python.langchain.com/docs/integrations/chat/openai/)
- Ask in the cohort channel — helping each other is encouraged! (but don't share your full solution)

---
*AI Engineer Ready — Cohort 1 | Assignment 1 | Option 3*